# Data Integrity: can we trust the data, and what drives revenue?

This notebook has three jobs:

1. **Build a clean revenue pipeline** (clean the tables, build one fact table, split revenue growth into drivers)
2. **Test the data for traps** (duplicates, cancelled bookings, ghost records, missing links) before we trust any number
3. **Investigate payments** (do invoices, payments and settlements agree?)

**Parts**

| Part | What happens |
|---|---|
| 1 | Setup: imports, paths, small helper functions |
| 2 | Cleaning functions for bookings, lines, invoices |
| 3 | Fact table: one row per invoice, with its test count |
| 4 | Unit economics and the driver split (patients x tests per patient x revenue per test) |
| 5 | Diagnostic helpers (discounts, collections, plan vs actual) |
| 6 | Run the pipeline |
| 7 | Which definition of "test volume" makes the plan equal +18%? |
| 8 | Ten data-trap checks, each printing PASS / FLAG / INFO |
| 9 | Payments detective work |

**How to run:** top to bottom, once. Set `DATA_DIR` in Part 1 first. Each cell is explained before it.

**What changed from the original script (so you are not surprised):**
- `input()` for the data folder is replaced by a plain variable `DATA_DIR`
- the big `main()` function is unrolled into normal cells so you can see every output
- the old "plan vs actual on invoiced tests" cell is left out, because the booked-basis version replaced it (see the `5_vs_18_beginner.ipynb` notebook)

## Part 1. Setup

### Imports and display options

- `numpy` (as `np`) for maths and `np.select`
- `pandas` (as `pd`) for tables
- `pd.options.display...` only changes how tables print: thousands separators, 3 decimals, wider tables

**What we used:** `import ... as ...`, `pd.options.display`.

In [8]:
from google.colab import drive
drive.mount('/content/drive/')


Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [9]:
import os
import numpy as np
import pandas as pd

pd.options.display.float_format = "{:,.3f}".format
pd.options.display.width = 200
pd.options.display.max_columns = 30

### Folder and period settings

- `DATA_DIR`: the folder holding the `*.csv.gz` files. **Change this** if yours is elsewhere
- `CUR` and `PRV`: the two periods we compare (current H1-2026 and the same months of last year). A tuple is a fixed pair of (start, end)
- `CRORE`: 1 crore = 10,000,000, so we can show rupees in crores

**What we used:** constants, tuples, scientific notation (`1e7`).

In [10]:
DATA_DIR = "/content/drive/MyDrive/merged"          # <- change if your folder is different

CUR = ("2026-01-01", "2026-06-30")   # current period (like-for-like vs same months last year)
PRV = ("2025-01-01", "2025-06-30")   # prior period
CRORE = 1e7

print(os.path.isdir(DATA_DIR))
print(os.listdir(DATA_DIR))

True
['orphans_results.csv.gz', 'orphans_collections.csv.gz', 'orphans_invoices.csv.gz', 'booking_lines.csv.gz', 'bookings.csv.gz', 'collections.csv.gz', 'invoices.csv.gz', 'plan_long.csv.gz', 'campaigns.csv.gz', 'payments.csv.gz']


### Two tiny helper functions

- `read(name)` loads `<name>.csv.gz` from `DATA_DIR`. pandas decompresses `.gz` by itself. `**kw` lets us pass extra options such as `usecols=[...]` straight to `read_csv`
- `norm_id(s)` makes IDs comparable. One table writes `INV2026090659`, another `INV-2026-090659`. We upper-case them and delete everything that is not a letter or digit

**What we used:** `def`, `**kwargs`, `os.path.join`, f-strings, `.str.upper()`, `.str.replace(regex)`.

In [11]:
def read(name, **kw):
    return pd.read_csv(os.path.join(DATA_DIR, f"{name}.csv.gz"), **kw)


def norm_id(s: pd.Series) -> pd.Series:
    """payments.invoice_id_raw shows 'INV2026090659' vs 'INV-2026-090659' -> strip punctuation, upper-case."""
    return s.astype("string").str.upper().str.replace(r"[^A-Z0-9]", "", regex=True)

## Part 2. Cleaning functions

Every function takes a table in and returns a cleaner table out, and **prints what it removed**. That printout is our first integrity check.

### `clean_bookings`

Steps inside:
1. Drop repeated `booking_id`s
2. Strip spaces from IDs and turn the two date columns into real dates (`errors="coerce"` turns bad dates into blanks instead of crashing)
3. Find **cancelled** bookings (`status_at_export` contains "cancel")
4. Find **superseded** bookings: when a booking is rescheduled, the new booking points back to the old one in `rescheduled_from`. The old one must not be counted again
5. Return only rows that are neither cancelled nor superseded

**What we used:** `drop_duplicates`, `.str.strip()`, `pd.to_datetime`, `.str.contains`, `.isin`, boolean masks with `~` (not) and `&` (and).

In [12]:
def clean_bookings(b: pd.DataFrame) -> pd.DataFrame:
    b = b.drop_duplicates("booking_id").copy()
    b["booking_id"] = b["booking_id"].str.strip()
    for c in ("created_at", "slot_at"):
        b[c] = pd.to_datetime(b[c], errors="coerce")

    # status_at_export is only populated for the 'nova' system (~5% of rows). Drop anything cancelled.
    cancelled = b["status_at_export"].astype("string").str.lower().str.contains("cancel", na=False)
    # a rescheduled booking points to the booking it replaced -> the original must not be double counted
    superseded = b["booking_id"].isin(b["rescheduled_from"].dropna())

    print(f"[bookings] {len(b):,} rows | cancelled dropped: {cancelled.sum():,} | "
          f"superseded-by-reschedule dropped: {superseded.sum():,}")
    print("[bookings] status_at_export values:", b["status_at_export"].value_counts(dropna=False).to_dict())
    return b[~cancelled & ~superseded]

### `clean_lines`

Keep only test lines that belong to a valid booking. About 0.1% of lines have no `test_code`. We keep them (they still carry a price and a sample) but count them so we know.

**What we used:** `.isin`, `.copy()`, `.isna().sum()`.

In [13]:
def clean_lines(l: pd.DataFrame, valid_ids: pd.Series) -> pd.DataFrame:
    l = l[l["booking_id"].isin(valid_ids)].copy()
    # test_code is missing on ~0.1% of lines: they still carry a price and a sample, so keep and flag them
    print(f"[lines] {len(l):,} rows | missing test_code kept: {l['test_code'].isna().sum():,}")
    return l

### `clean_invoices`

- Drop duplicate invoices
- Convert date and amount to proper types (bad values become blanks)
- Add `invoice_key` (the normalised ID from `norm_id`)
- Drop rows with a bad date or amount, but only **report** non-positive amounts

**What we used:** `pd.to_numeric(errors="coerce")`, `|` (or) between two conditions.

In [14]:
def clean_invoices(inv: pd.DataFrame) -> pd.DataFrame:
    inv = inv.drop_duplicates("invoice_id").copy()
    inv["invoice_date"] = pd.to_datetime(inv["invoice_date"], errors="coerce")
    inv["amount"] = pd.to_numeric(inv["amount"], errors="coerce")
    inv["invoice_key"] = norm_id(inv["invoice_id"])
    bad = inv["invoice_date"].isna() | inv["amount"].isna()
    print(f"[invoices] {len(inv):,} rows | bad date/amount dropped: {bad.sum():,} | "
          f"non-positive amounts (kept): {(inv['amount'] <= 0).sum():,}")
    return inv[~bad]

## Part 3. The fact table (one row per invoice)

`build_fact` joins invoices with their booking details and adds:

- `samples` = number of test lines on the booking (counted with `groupby().size()`)
- `period` = `CUR`, `PRV` or `OUT` (outside both periods), chosen by `np.select`
- `month` = e.g. `2026-03`

An **inner merge** keeps only invoices whose booking survived cleaning. Invoices on cancelled, superseded or unknown bookings fall out, and the function prints how many.

**What we used:** `groupby().size()`, `merge(how="inner")`, `.map()`, `.fillna()`, `np.select(conditions, labels, default)`, `.between()`, `.dt.to_period("M")`.

In [15]:
def build_fact(inv, bookings, lines) -> pd.DataFrame:
    samples = lines.groupby("booking_id").size().rename("samples")
    f = inv.merge(bookings[["booking_id", "source_system", "tests_total", "home_fee"]],
                  on="booking_id", how="inner")            # invoices whose booking was cancelled/superseded/unknown fall out
    print(f"[fact] invoices kept {len(f):,} of {len(inv):,} (dropped {len(inv) - len(f):,} with no valid booking)")
    f["samples"] = f["booking_id"].map(samples).fillna(0)
    print(f"[fact] invoices with 0 lines: {(f['samples'] == 0).sum():,}")

    f["period"] = np.select(
        [f["invoice_date"].between(*CUR), f["invoice_date"].between(*PRV)], ["CUR", "PRV"], default="OUT")
    f["month"] = f["invoice_date"].dt.to_period("M").astype(str)
    return f

## Part 4. Unit economics and the revenue driver split

### `summarise`

For each period (and optionally each city or channel) it computes:

- **Revenue** = sum of invoice amounts
- **Patients** = distinct `patient_id`s (`nunique`)
- **Samples** = tests
- Then three ratios: revenue per patient, revenue per sample, samples per patient

**What we used:** `groupby(...).agg(name=(column, function))`, `nunique`, column division.

In [16]:
def summarise(f: pd.DataFrame, by=None) -> pd.DataFrame:
    keys = ["period"] + ([by] if by else [])
    g = f[f["period"] != "OUT"].groupby(keys).agg(
        Revenue=("amount", "sum"),
        Patients=("patient_id", "nunique"),     # distinct customers
        Samples=("samples", "sum"))             # tests
    g["Rev_per_Patient"] = g["Revenue"] / g["Patients"]
    g["Rev_per_Sample"] = g["Revenue"] / g["Samples"]
    g["Samples_per_Patient"] = g["Samples"] / g["Patients"]
    return g

### `growth_table`

Puts previous and current side by side and adds absolute change and % change.

**What we used:** `.loc["PRV"]` (pick the PRV block of a two-level index), `pd.DataFrame({...})`.

In [17]:
def growth_table(s: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame({"Previous": s.loc["PRV"], "Current": s.loc["CUR"]})
    out["Abs Change"] = out["Current"] - out["Previous"]
    out["% Change"] = out["Current"] / out["Previous"] - 1
    return out

### `decompose`: why did revenue grow?

Revenue can be written as a product:

**Revenue = Patients x (Samples per patient) x (Revenue per sample)**

So growth can come from more patients, more tests per patient (**utilisation**), or a higher price per test (**realisation**). Because these multiply, we cannot just add percentages. We use **logs**: the log of a product is the sum of the logs, so each driver gets a share of the revenue change that adds up exactly (the `assert` proves it).

**What we used:** `np.log`, `np.expm1` (turns a log back into a % growth), `pd.Series`, `assert`, `np.isclose`.

In [18]:
def decompose(prv: pd.Series, cur: pd.Series) -> pd.DataFrame:
    P0, P1 = prv["Patients"], cur["Patients"]
    U0, U1 = prv["Samples_per_Patient"], cur["Samples_per_Patient"]
    V0, V1 = prv["Rev_per_Sample"], cur["Rev_per_Sample"]
    dR = cur["Revenue"] - prv["Revenue"]
    logs = pd.Series({"Patient growth": np.log(P1 / P0),
                      "Utilization (samples/patient)": np.log(U1 / U0),
                      "Realization (revenue/sample)": np.log(V1 / V0)})
    impact = logs / logs.sum() * dR if logs.sum() != 0 else logs * 0
    out = pd.DataFrame({"Growth %": np.expm1(logs), "Impact": impact, "Share": impact / dR if dR else np.nan})
    assert np.isclose(out["Impact"].sum(), dR), "decomposition does not reconcile"
    return out

### `insights`

Turns the numbers into plain sentences: did revenue outgrow patients, did utilisation rise, did realisation improve, and which driver is biggest. It returns a **list of strings**.

**What we used:** tuple unpacking (`gP, gU, gV = ...`), `idxmax()`, conditional text (`"a" if cond else "b"`), f-string formats (`:+.1%`, `:,.0f`).

In [19]:
def insights(prv: pd.Series, cur: pd.Series, d: pd.DataFrame, label="Total") -> list[str]:
    gR = cur["Revenue"] / prv["Revenue"] - 1
    gP, gU, gV = d["Growth %"]
    top = d["Share"].idxmax()
    return [
        f"[{label}] Revenue {gR:+.1%} vs patients {gP:+.1%} -> "
        + ("revenue outgrew customers." if gR > gP else "revenue lagged customer growth."),
        f"[{label}] Samples/patient {prv['Samples_per_Patient']:.2f} -> {cur['Samples_per_Patient']:.2f} ({gU:+.1%}): "
        + ("customers consumed more services." if gU > 0 else "utilization fell."),
        f"[{label}] Revenue/sample INR {prv['Rev_per_Sample']:,.0f} -> {cur['Rev_per_Sample']:,.0f} ({gV:+.1%}): "
        + ("realization improved." if gV > 0 else "realization weakened (check discounts/waivers/mix)."),
        f"[{label}] Primary driver: {top} ({d.loc[top, 'Share']:.0%} of incremental revenue).",
    ]

## Part 5. Diagnostic helpers

### `realization_bridge`: why did revenue per sample move?

Compares, per period and per sample: revenue, list value of the tests, home-collection fees and campaign waivers. If revenue per sample fell, this shows whether discounts or mix caused it.

**What we used:** `groupby().sum()`, `.map()` to look up values from another table, `.T` (transpose, so periods become columns), a loop that builds `*_per_Sample` columns.

In [20]:
def realization_bridge(f, lines, campaigns) -> pd.DataFrame:
    """Why did revenue/sample move?  list price of tests, home fee, campaign waivers."""
    list_amt = lines.groupby("booking_id")["list_price_rs"].sum().rename("list_total")
    waiver = campaigns.drop_duplicates("booking_id").set_index("booking_id")["waiver_rs"]
    x = f[f["period"] != "OUT"].copy()
    x["list_total"] = x["booking_id"].map(list_amt)
    x["waiver"] = x["booking_id"].map(waiver).fillna(0)
    g = x.groupby("period").agg(Samples=("samples", "sum"), Revenue=("amount", "sum"),
                                List_Value=("list_total", "sum"), Home_Fees=("home_fee", "sum"),
                                Waivers=("waiver", "sum"))
    for c in ("Revenue", "List_Value", "Home_Fees", "Waivers"):
        g[f"{c}_per_Sample"] = g[c] / g["Samples"]
    return g.T

### `reconcile_invoices`

An invoice should equal **tests_total + home_fee - waiver**. This counts how many invoices match to within INR 1.

**What we used:** column arithmetic, `.abs()`, `.mean()` on True/False values (gives the share that is True).

In [21]:
def reconcile_invoices(f, campaigns) -> None:
    """invoice amount should ~ tests_total + home_fee - waiver."""
    waiver = campaigns.drop_duplicates("booking_id").set_index("booking_id")["waiver_rs"]
    exp = f["tests_total"] + f["home_fee"] - f["booking_id"].map(waiver).fillna(0)
    ok = (f["amount"] - exp).abs() <= 1
    print(f"[check] invoices matching tests_total + home_fee - waiver: {ok.mean():.1%} "
          f"({(~ok).sum():,} mismatches)")

### `collections_check`: billed vs collected cash

`payments` can have several rows per invoice (including failed retries). We keep only `captured` payments, one per invoice, and compare cash collected to amount billed.

**What we used:** boolean filtering, `drop_duplicates`, left `merge`, `groupby().agg()`.

In [22]:
def collections_check(f, payments) -> pd.DataFrame:
    """Cash collected vs billed (payments: several rows per invoice incl. failed retries)."""
    p = payments[payments["payment_status"] == "captured"].copy()
    p["invoice_key"] = norm_id(p["invoice_id"])
    p = p.drop_duplicates("invoice_key")                      # one captured payment per invoice
    x = f[f["period"] != "OUT"].merge(p[["invoice_key", "amount_rs"]], on="invoice_key", how="left")
    g = x.groupby("period").agg(Billed=("amount", "sum"), Collected=("amount_rs", "sum"))
    g["Collection_%"] = g["Collected"] / g["Billed"]
    return g

### `plan_vs_actual` (invoiced-test version)

Monthly 2026 tests vs plan by city. (Note: the plan was later found to be on a **booked** basis, so Part 7 and the other notebook redo this properly. We keep the helper because `main` used it.)

**What we used:** `pd.concat(axis=1)`, `dropna(subset=...)`.

In [23]:
def plan_vs_actual(f, plan) -> pd.DataFrame:
    act = f[f["month"].str.startswith("2026")].groupby(["city", "month"])["samples"].sum().rename("actual")
    p = plan.set_index(["city", "month"])["target"]
    out = pd.concat([p, act], axis=1).dropna(subset=["target"])
    out["attainment"] = out["actual"] / out["target"]
    return out

## Part 6. Run the pipeline

The original `main()` did all of this inside one function. Here each step is its own cell so you can look at the output.

### Load and clean everything

The order matters: bookings first, because lines are filtered by valid booking IDs.

**What we used:** calling the functions we defined, `usecols` to load only needed columns.

In [24]:
bookings = clean_bookings(read("bookings"))
lines = clean_lines(read("booking_lines", usecols=["booking_id", "test_code", "test_price", "list_price_rs"]),
                    bookings["booking_id"])
inv = clean_invoices(read("invoices"))
campaigns = read("campaigns")
payments = read("payments", usecols=["invoice_id", "payment_status", "amount_rs"])
plan = read("plan_long")

/tmp/ipykernel_9713/2968243182.py:2: DtypeWarning: Columns (8,11,12,13) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(os.path.join(DATA_DIR, f"{name}.csv.gz"), **kw)


[bookings] 568,007 rows | cancelled dropped: 0 | superseded-by-reschedule dropped: 1,500
[bookings] status_at_export values: {nan: 541464, 'booked': 25043, 'rescheduled': 1500}
[lines] 1,311,662 rows | missing test_code kept: 1,310
[invoices] 470,776 rows | bad date/amount dropped: 0 | non-positive amounts (kept): 0


### Build the fact table

**What we used:** `build_fact`, `.min().date()` to print a clean date.

In [25]:
f = build_fact(inv, bookings, lines)
print("\ninvoice_date range:", f["invoice_date"].min().date(), "->", f["invoice_date"].max().date())

[fact] invoices kept 469,534 of 470,776 (dropped 1,242 with no valid booking)
[fact] invoices with 0 lines: 0

invoice_date range: 2024-12-30 -> 2026-06-30


### Unit economics, current vs previous

**What we used:** `summarise` then `growth_table`.

In [26]:
# ---- company level
s = summarise(f)
print("\n=== UNIT ECONOMICS ===")
ue = growth_table(s)
print(ue)


=== UNIT ECONOMICS ===
                           Previous         Current     Abs Change  % Change
Revenue             239,582,707.000 266,119,871.000 26,537,164.000     0.111
Patients                 84,528.000      87,698.000      3,170.000     0.038
Samples                 361,751.000     379,423.000     17,672.000     0.049
Rev_per_Patient           2,834.359       3,034.503        200.144     0.071
Rev_per_Sample              662.286         701.380         39.094     0.059
Samples_per_Patient           4.280           4.326          0.047     0.011


### Revenue growth decomposition

We pass the PRV row and the CUR row of `s` into `decompose`, then show the impact in crores.

**What we used:** `.assign(new_col=...)`, `.loc`.

In [27]:
d = decompose(s.loc["PRV"], s.loc["CUR"])
d_cr = d.assign(Impact_Rs_cr=d["Impact"] / CRORE)
print("\n=== REVENUE GROWTH DECOMPOSITION ===")
print(d_cr)


=== REVENUE GROWTH DECOMPOSITION ===
                               Growth %         Impact  Share  Impact_Rs_cr
Patient growth                    0.038  9,300,486.109  0.350         0.930
Utilization (samples/patient)     0.011  2,748,316.687  0.104         0.275
Realization (revenue/sample)      0.059 14,488,361.204  0.546         1.449


### Insight sentences

**What we used:** a `for` loop over the list returned by `insights`.

In [28]:
print("\n=== INSIGHTS ===")
for line in insights(s.loc["PRV"], s.loc["CUR"], d):
    print("-", line)


=== INSIGHTS ===
- [Total] Revenue +11.1% vs patients +3.8% -> revenue outgrew customers.
- [Total] Samples/patient 4.28 -> 4.33 (+1.1%): customers consumed more services.
- [Total] Revenue/sample INR 662 -> 701 (+5.9%): realization improved.
- [Total] Primary driver: Realization (revenue/sample) (55% of incremental revenue).


### Driver mix by city

We repeat the decomposition for every city and collect the **share of growth** each driver explains. The `if` guard skips a city that is missing in one of the periods.

**What we used:** a `for` loop over a MultiIndex level (`get_level_values`), a dictionary of dictionaries, dict unpacking (`**{...}`), `pd.DataFrame(rows).T`.

In [29]:
# ---- by city
sc = summarise(f, by="city")
rows = {}
for city in sc.index.get_level_values("city").unique():
    if ("PRV", city) in sc.index and ("CUR", city) in sc.index:
        r = decompose(sc.loc[("PRV", city)], sc.loc[("CUR", city)])
        rows[city] = {"Rev growth": sc.loc[("CUR", city), "Revenue"] / sc.loc[("PRV", city), "Revenue"] - 1,
                      **{f"{k} share": v for k, v in r["Share"].items()}}
print("\n=== DRIVER MIX BY CITY ===")
print(pd.DataFrame(rows).T)


=== DRIVER MIX BY CITY ===
           Rev growth  Patient growth share  Utilization (samples/patient) share  Realization (revenue/sample) share
Ahmedabad       0.172                 0.615                                0.038                               0.347
Bengaluru       0.147                 0.557                                0.066                               0.376
Chennai         0.042                -1.491                                1.088                               1.403
Hyderabad       0.152                 0.570                               -0.036                               0.465
Jaipur         -0.071                 1.922                               -0.060                              -0.862
Pune            0.148                 0.577                                0.024                               0.398


### Diagnostics: discounts, collections, plan and reconciliation

Four quick checks:
1. realisation bridge (list price vs realised)
2. billed vs cash collected
3. plan vs actual (last 12 rows)
4. how many invoices match tests + fee - waiver

In [30]:
print("\n=== REALIZATION BRIDGE (per sample) ===");  print(realization_bridge(f, lines, campaigns))
print("\n=== COLLECTIONS ===");                       print(collections_check(f, payments))


=== REALIZATION BRIDGE (per sample) ===
period                            CUR             PRV
Samples                   379,423.000     361,751.000
Revenue               266,119,871.000 239,582,707.000
List_Value            260,914,776.000 230,892,539.000
Home_Fees               4,937,250.000   8,455,650.000
Waivers                 6,403,650.000           0.000
Revenue_per_Sample            701.380         662.286
List_Value_per_Sample         687.662         638.264
Home_Fees_per_Sample           13.013          23.374
Waivers_per_Sample             16.877           0.000

=== COLLECTIONS ===
                Billed       Collected  Collection_%
period                                              
CUR    266,119,871.000 249,849,945.000         0.939
PRV    239,582,707.000 230,225,095.000         0.961


In [31]:
print("\n=== PLAN VS ACTUAL (2026 test volume) ===");  print(plan_vs_actual(f, plan).tail(12))
print();  reconcile_invoices(f, campaigns)


=== PLAN VS ACTUAL (2026 test volume) ===
                   target  actual  attainment
city      month                              
Ahmedabad 2026-05   15019   11599       0.772
Bengaluru 2026-05   19883   14719       0.740
Chennai   2026-05   15512   10183       0.656
Hyderabad 2026-05   14001   10275       0.734
Jaipur    2026-05    8484    5133       0.605
Pune      2026-05   11779    8237       0.699
Ahmedabad 2026-06   15419   11791       0.765
Bengaluru 2026-06   19655   14051       0.715
Chennai   2026-06   15646   10067       0.643
Hyderabad 2026-06   13693   10902       0.796
Jaipur    2026-06    8038    5283       0.657
Pune      2026-06   11818    8923       0.755

[check] invoices matching tests_total + home_fee - waiver: 90.9% (42,712 mismatches)


## Part 7. Which definition of "test volume" makes the plan equal +18%?

The plan says +18% on H1-25, but invoiced tests grew only about +5%. Maybe the plan counts something else. We test four definitions of "test":

| Basis | Meaning |
|---|---|
| Booked (created_at) | tests booked, placed in the month the booking was made |
| Booked (slot_at) | tests booked, placed in the month of the service slot |
| Collected (slot_at) | only visits where the sample was actually collected |
| Invoiced (invoice_date) | tests on invoices |

The basis whose implied plan growth is closest to 18% is probably the one the plan used. (Finding: it is **booked** tests.)

### Setup and the bookings with test counts

- `CLAIMED_GROWTH` is set here (it lived in a cell we dropped)
- `collected` is True when the collection status is "collected" (not "no_show")

**What we used:** list comprehension, `.drop_duplicates().set_index()["col"]` (a lookup table), `.map()`, `.eq()`.

In [32]:
CLAIMED_GROWTH = 0.18

import pandas as pd

PRV_M = [f"2025-{m:02d}" for m in range(1, 7)]
CUR_M = [f"2026-{m:02d}" for m in range(1, 7)]

# --- 1. bookings with test counts -----------------------------------------------------------------
bk = clean_bookings(read("bookings", usecols=["booking_id", "city", "created_at", "slot_at",
                                              "status_at_export", "rescheduled_from"]))
tests = read("booking_lines", usecols=["booking_id"]).groupby("booking_id").size().rename("tests")
bk["tests"] = bk["booking_id"].map(tests).fillna(0)

coll = read("collections", usecols=["booking_id", "status"]).drop_duplicates("booking_id").set_index("booking_id")["status"]
bk["collected"] = bk["booking_id"].map(coll).eq("collected")          # status: collected / no_show

/tmp/ipykernel_9713/2968243182.py:2: DtypeWarning: Columns (11,12) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(os.path.join(DATA_DIR, f"{name}.csv.gz"), **kw)


[bookings] 568,007 rows | cancelled dropped: 0 | superseded-by-reschedule dropped: 1,500
[bookings] status_at_export values: {nan: 541464, 'booked': 25043, 'rescheduled': 1500}


### A function and a dictionary of four series

`monthly(...)` returns tests per (city, month) for one definition. We store the four versions in a dictionary so we can loop over them.

**What we used:** function with a docstring, `groupby` on a column plus a computed Series, a dictionary of Series.

In [33]:
def monthly(df, date_col, tests_col, basis):
    """tests by (city, month) for one basis"""
    m = df[date_col].dt.to_period("M").astype(str).rename("month")
    return df.groupby(["city", m])[tests_col].sum().rename(basis)


series = {
    "Booked (created_at)":    monthly(bk, "created_at", "tests", "Booked (created_at)"),
    "Booked (slot_at)":       monthly(bk, "slot_at", "tests", "Booked (slot_at)"),
    "Collected (slot_at)":    monthly(bk[bk["collected"]], "slot_at", "tests", "Collected (slot_at)"),
    "Invoiced (invoice_date)": f.groupby(["city", "month"])["samples"].sum().rename("Invoiced (invoice_date)"),
}

### Compare the four bases

For each basis we compute H1-25 and H1-26 totals, actual growth, plan-implied growth, and attainment. Then `idxmin` on the absolute difference to 18% finds the best fit.

**What we used:** `for name, s in dict.items()`, `isin` on an index level, `.abs().idxmin()`.

In [34]:
# --- 2. total-level comparison -------------------------------------------------------------------
plan_h1 = plan[plan["month"].isin(CUR_M)].groupby("city")["target"].sum()
rows = {}
for name, s in series.items():
    mo = s.index.get_level_values("month")
    h25 = s[mo.isin(PRV_M)].groupby("city").sum()
    h26 = s[mo.isin(CUR_M)].groupby("city").sum()
    rows[name] = {
        "H1-25 actual": h25.sum(), "H1-26 actual": h26.sum(), "H1-26 plan": plan_h1.sum(),
        "Actual growth": h26.sum() / h25.sum() - 1,
        "Plan growth (implied)": plan_h1.sum() / h25.sum() - 1,
        "Attainment": h26.sum() / plan_h1.sum(),
    }
res = pd.DataFrame(rows).T
res["Plan vs claimed"] = res["Plan growth (implied)"] - CLAIMED_GROWTH
print(f"=== WHICH BASIS MAKES THE PLAN ~ +{CLAIMED_GROWTH:.0%}? ===")
print(res.to_string(float_format=lambda x: f"{x:,.3f}"))

best = res["Plan vs claimed"].abs().idxmin()
print(f"\nClosest to {CLAIMED_GROWTH:.0%}: {best} "
      f"(plan {res.loc[best, 'Plan growth (implied)']:+.1%}, actual {res.loc[best, 'Actual growth']:+.1%}, "
      f"attainment {res.loc[best, 'Attainment']:.0%})")
print("Fits within 1pp:", abs(res.loc[best, "Plan vs claimed"]) <= 0.01)

=== WHICH BASIS MAKES THE PLAN ~ +18%? ===
                         H1-25 actual  H1-26 actual  H1-26 plan  Actual growth  Plan growth (implied)  Attainment  Plan vs claimed
Booked (created_at)       436,512.000   451,786.000 515,740.000          0.035                  0.182       0.876            0.002
Booked (slot_at)          436,302.000   458,002.000 515,740.000          0.050                  0.182       0.888            0.002
Collected (slot_at)       361,475.000   379,165.000 515,740.000          0.049                  0.427       0.735            0.247
Invoiced (invoice_date)   361,751.000   379,423.000 515,740.000          0.049                  0.426       0.736            0.246

Closest to 18%: Booked (created_at) (plan +18.2%, actual +3.5%, attainment 88%)
Fits within 1pp: True


### City detail on the best-fit basis, and booked-to-invoiced leakage

- The city table is rebuilt for the winning basis
- Conversion = invoiced tests / booked tests, compared between H1-25 and H1-26. A falling conversion means more booked tests are not becoming invoices

**What we used:** dict comprehension, `isin` on an index level, ratio of sums.

In [35]:
# --- 3. city detail on the best-fit basis --------------------------------------------------------
s = series[best]; mo = s.index.get_level_values("month")
city = pd.DataFrame({"H1-25 actual": s[mo.isin(PRV_M)].groupby("city").sum(),
                     "H1-26 actual": s[mo.isin(CUR_M)].groupby("city").sum(), "H1-26 plan": plan_h1})
city["Actual growth"] = city["H1-26 actual"] / city["H1-25 actual"] - 1
city["Plan growth"] = city["H1-26 plan"] / city["H1-25 actual"] - 1
city["Attainment"] = city["H1-26 actual"] / city["H1-26 plan"]
print(f"\n=== BY CITY - {best} ===")
print(city.to_string(float_format=lambda x: f"{x:,.3f}"))

# --- 4. leakage: booked -> invoiced conversion, H1-25 vs H1-26 ---------------------------------------
b, i = series["Booked (created_at)"], series["Invoiced (invoice_date)"]
conv = {p: i[i.index.get_level_values("month").isin(ms)].sum() / b[b.index.get_level_values("month").isin(ms)].sum()
        for p, ms in (("H1-25", PRV_M), ("H1-26", CUR_M))}
print(f"\nInvoiced / booked tests:  H1-25 {conv['H1-25']:.1%}  ->  H1-26 {conv['H1-26']:.1%}")


=== BY CITY - Booked (created_at) ===
           H1-25 actual  H1-26 actual  H1-26 plan  Actual growth  Plan growth  Attainment
city                                                                                     
Ahmedabad         76983         83864       90773          0.089        0.179       0.924
Bengaluru        100085        106175      120085          0.061        0.200       0.884
Chennai           80363         79723       93912         -0.008        0.169       0.849
Hyderabad         73194         77631       86421          0.061        0.181       0.898
Jaipur            45111         39713       52372         -0.120        0.161       0.758
Pune              60776         64680       72177          0.064        0.188       0.896

Invoiced / booked tests:  H1-25 82.9%  ->  H1-26 84.0%


> **Where is the full "plan vs actual on booked tests" analysis?** In the earlier notebook `5_vs_18_beginner.ipynb`. It continues straight from this finding.

## Part 8. Data trap checks

Before trusting any number we test what the data dictionary *claims*. Each check prints one of:

- **PASS**: the claim is confirmed, or the pipeline already handles the risk
- **FLAG**: a problem exists in the data. Act on it
- **INFO**: context you must state when reporting numbers

Results are collected in a list called `report` and shown as a table at the end.

### Settings and a mini reporting tool

- `CENSOR_DAYS`: results younger than 14 days are treated as "not yet due"
- `EXPECT`: row counts stated in the dictionary, to compare against
- `add(trap, status, evidence)` appends one result and prints it

**What we used:** dict, list of dicts, a function that updates a global list.

In [36]:
import numpy as np
import pandas as pd

CENSOR_DAYS = 14                       # results younger than this are treated as "not yet due"
EXPECT = dict(bookings=568_007, collections=549_598, gap=18_409, lines=1_315_204,
              nova=26_543, rescheduled=1_500, orphan_results=5_401, missing_code_pct=0.0010)

report = []
def add(trap, status, evidence):
    report.append({"Trap": trap, "Status": status, "Evidence": evidence})
    print(f"\n[{status}] {trap}\n    {evidence}")

### Helpers to find columns by name

The dictionary only *hints* at some column names. These helpers search the column list so the script does not break if a name differs slightly.

- `header(name)` reads 0 rows (`nrows=0`), so it is instant and returns just the column names
- `pick(cols, *names)` returns the first column that matches one of the names, ignoring case
- `contains(cols, key)` returns the first column whose name contains a keyword

**What we used:** `nrows=0`, `next(generator, default)`, `*args`, `.lower()`.

In [37]:
def header(name):
    return list(read(name, nrows=0).columns)

def pick(cols, *names):
    low = {c.lower(): c for c in cols}
    return next((low[n.lower()] for n in names if n.lower() in low), None)

def contains(cols, key, exclude=()):
    return next((c for c in cols if key in c.lower() and c not in exclude), None)

### Detect the columns, then print what we found

If a guess is wrong you can correct the variable here (`SRC`, `NAME`, `IS_NEW`, `RES`, `LID`, `PV`, `FLAGC`).

**What we used:** `or` as "use the second option if the first is None".

In [38]:
bcols, lcols, ccols = header("bookings"), header("booking_lines"), header("collections")
SRC    = pick(bcols, "source") or contains(bcols, "source", exclude=("source_system",))
NAME   = contains(bcols, "patient_name")
IS_NEW = contains(bcols, "is_new")
RES    = pick(lcols, "result_ts")
LID    = contains(lcols, "line_id")
PV     = pick(lcols, "price_vs_list_rs")
FLAGC  = pick(lcols, "test_code_missing")
print("DETECTED  source:", SRC, "| patient name:", NAME, "| is_new:", IS_NEW,
      "| result_ts:", RES, "| line id:", LID, "| price_vs_list:", PV, "| code-missing flag:", FLAGC)

DETECTED  source: source | patient name: patient_name | is_new: patient.is_new | result_ts: result_ts | line id: None | price_vs_list: price_vs_list_rs | code-missing flag: test_code_missing


### Load the tables

- `dict.fromkeys(want)` removes duplicate names but **keeps order**
- we only request columns that actually exist (`if c in bcols`)
- dates are converted right after loading

**What we used:** `usecols` with a list comprehension, `low_memory=False`, `pd.to_datetime`.

In [39]:
want = ["booking_id", "source_system", "channel", "status_at_export", "rescheduled_from", "created_at",
        "slot_at", "city", "tests_total", "home_fee"] + [c for c in (SRC, NAME, IS_NEW) if c]
bk = read("bookings", usecols=[c for c in dict.fromkeys(want) if c in bcols], low_memory=False)
for c in ("created_at", "slot_at"):
    bk[c] = pd.to_datetime(bk[c], errors="coerce")

lwant = ["booking_id", "test_code", "test_price", "list_price_rs"] + [c for c in (RES, LID, PV, FLAGC) if c]
ln = read("booking_lines", usecols=[c for c in dict.fromkeys(lwant) if c in lcols], low_memory=False)
if RES:
    ln[RES] = pd.to_datetime(ln[RES], errors="coerce")

col = read("collections", usecols=[c for c in ["booking_id", "status", "collected_at", "phlebotomist_id", "channel"]
                                   if c in ccols])
inv = read("invoices", usecols=["invoice_id", "invoice_date", "booking_id", "patient_id", "amount"])
inv["invoice_date"] = pd.to_datetime(inv["invoice_date"], errors="coerce")
camp = read("campaigns", usecols=["booking_id", "waiver_rs"]).drop_duplicates("booking_id")
orph = read("orphans_results", usecols=["booking_id", "test_code", "result_ts"])

### Derived flags reused by every check

| New column | Meaning |
|---|---|
| `is_nova` | booking came from the new "nova" system (vs legacy) |
| `superseded` | this booking was replaced by a reschedule |
| `tests` | number of test lines (the *grain* of volume) |
| `coll_status` | collected / no_show / unknown (no record) |
| `invoiced` | has at least one invoice |
| `month` | month of the slot |

`bk_idx` is `bk` indexed by `booking_id` for fast lookups later.

**What we used:** `.str.contains`, `set(...)` for fast membership tests, `.map` with a Series, `.set_index`.

In [40]:
bk["is_nova"]    = bk["source_system"].astype(str).str.lower().str.contains("nova")
orig_ids         = set(bk["rescheduled_from"].dropna())
bk["superseded"] = bk["booking_id"].isin(orig_ids)                      # original of a reschedule
bk["tests"]      = bk["booking_id"].map(ln.groupby("booking_id").size()).fillna(0)   # GRAIN: lines
bk["coll_status"] = bk["booking_id"].map(col.drop_duplicates("booking_id").set_index("booking_id")["status"]).fillna("unknown")
bk["invoiced"]   = bk["booking_id"].isin(set(inv["booking_id"]))
bk["month"]      = bk["slot_at"].dt.to_period("M").astype(str)
bk_idx = bk.set_index("booking_id")

### Check 1. "collections" is about sample collection, not money

If this is right, `collected_at` is blank only for no-shows, and the money columns live in `payments`, never in `collections`.

**What we used:** `.all()` on boolean Series, `groupby().apply(lambda ...)`, `add(...)`.

In [41]:
ns = col["status"].eq("no_show")
clean_pat = col.loc[ns, "collected_at"].isna().all() and col.loc[~ns, "collected_at"].notna().all()
phleb = (col.groupby("channel")["phlebotomist_id"].apply(lambda s: s.isna().mean()).round(3).to_dict()
         if {"channel", "phlebotomist_id"} <= set(col.columns) else "n/a")
money_cols = [c for c in header("payments") if any(k in c.lower() for k in ("amount", "settle", "fees", "utr"))]
add("1. 'collections' is phlebotomy, not money", "PASS" if clean_pat else "FLAG",
    f"status counts {col['status'].value_counts().to_dict()}; no_show rows {ns.sum():,}; "
    f"collected_at null on no_show only: {clean_pat}; phlebotomist null share by channel {phleb}. "
    f"Money lives in payments ({', '.join(money_cols[:6])}...) - never join collections for revenue/cash.")


[PASS] 1. 'collections' is phlebotomy, not money
    status counts {'collected': 470426, 'no_show': 79172}; no_show rows 79,172; collected_at null on no_show only: True; phlebotomist null share by channel {'clinic': 1.0, 'home': 0.0}. Money lives in payments (settlement_id, amount_rs, utr, settlement_status, settlement_amount_rs, fees_rs...) - never join collections for revenue/cash.


### Check 2. Two booking systems

Only the nova system fills `status_at_export`, patient name and `is_new`. So the number of non-blank values in those columns should equal the number of nova bookings (about 26.5k). We also check that the `BK-` ID prefix does **not** reliably tell the system, so always split on `source_system`.

**What we used:** dict of counts, `all(...)` over a list, `value_counts().to_dict()`.

In [42]:
nova_n = int(bk["is_nova"].sum())
prefix_clash = int((bk["booking_id"].str.startswith("BK-") == bk["is_nova"]).sum())
nn = {"patient name null": int(bk[NAME].isna().sum()) if NAME else None,
      "status_at_export non-null": int(bk["status_at_export"].notna().sum()),
      "is_new non-null": int(bk[IS_NEW].notna().sum()) if IS_NEW else None}
vals = [v for v in nn.values() if v is not None]
consistent = all(v == nova_n for v in vals)
add("2. Two booking systems", "PASS" if consistent and nova_n == EXPECT["nova"] and prefix_clash == 0 else "FLAG",
    f"source_system counts {bk['source_system'].value_counts().to_dict()}; nova {nova_n:,} "
    f"({nova_n / len(bk):.1%}, expected {EXPECT['nova']:,}); rows where BK- prefix disagrees with source_system: "
    f"{prefix_clash:,}; nova-only fields {nn}. Always split on source_system, never on the ID prefix.")


[PASS] 2. Two booking systems
    source_system counts {'legacy': 541464, 'nova': 26543}; nova 26,543 (4.7%, expected 26,543); rows where BK- prefix disagrees with source_system: 0; nova-only fields {'patient name null': 26543, 'status_at_export non-null': 26543, 'is_new non-null': 26543}. Always split on source_system, never on the ID prefix.


### Check 3. Nova fields look unreliable

A booking cannot sensibly be `channel = home` **and** `source = walkin`. We compare how often that happens in nova vs legacy. If nova is more than 5x worse, we flag its source field.

**What we used:** `&` between boolean masks, `pd.crosstab`.

In [43]:
if SRC:
    walk = bk[SRC].astype(str).str.lower().eq("walkin")
    home = bk["channel"].astype(str).str.lower().eq("home")
    n_nova, n_leg = int((bk.is_nova & walk & home).sum()), int((~bk.is_nova & walk & home).sum())
    r_nova = n_nova / max(nova_n, 1); r_leg = n_leg / max(len(bk) - nova_n, 1)
    bad = n_nova > 0 and r_nova > 5 * r_leg
    add("3. Nova source/channel contradiction", "FLAG" if bad else "PASS",
        f"channel=home & {SRC}=walkin: nova {n_nova:,} ({r_nova:.1%} of nova) vs legacy {n_leg:,} ({r_leg:.2%}). "
        f"{'Do not use the nova source field without validation.' if bad else 'No contradiction visible.'}")
    print(pd.crosstab(bk.loc[bk.is_nova, "channel"], bk.loc[bk.is_nova, SRC]))
else:
    add("3. Nova source/channel contradiction", "INFO", "no source column found in bookings - set SRC manually.")


[PASS] 3. Nova source/channel contradiction
    channel=home & source=walkin: nova 1,703 (6.4% of nova) vs legacy 30,702 (5.67%). No contradiction visible.
source    app  phone  walkin
channel                     
clinic   8378   5087    2446
home     5685   3244    1703


### Check 4. Reschedules double count

A rescheduled booking has the same tests as its original. If both are counted, volume doubles for that visit. We verify: each replacement points to a real original, test counts match, and `clean_bookings` removes the originals.

**What we used:** `set.isdisjoint`, comparing arrays with `.values`, calling a function we wrote earlier.

In [44]:
new = bk[bk["rescheduled_from"].notna()]
orig_tests = bk_idx["tests"]
same_tests = (new["tests"].values == new["rescheduled_from"].map(orig_tests).values).mean() if len(new) else np.nan
inv_on_orig = inv[inv["booking_id"].isin(orig_ids)]
cleaned = clean_bookings(bk[["booking_id", "status_at_export", "rescheduled_from", "created_at", "slot_at"]].copy())
handled = orig_ids.isdisjoint(set(cleaned["booking_id"]))
add("4. Reschedule double counting", "PASS" if handled else "FLAG",
    f"{len(new):,} bookings carry rescheduled_from (expected {EXPECT['rescheduled']:,}); originals found in bookings "
    f"{new['rescheduled_from'].isin(bk['booking_id']).mean():.0%}; replacement has the same test count as its original "
    f"in {same_tests:.0%} of pairs; originals hold {int(bk.loc[bk.superseded, 'tests'].sum()):,} lines "
    f"({bk.loc[bk.superseded, 'tests'].sum() / bk['tests'].sum():.2%} of volume) and {len(inv_on_orig):,} invoices "
    f"(INR {inv_on_orig['amount'].sum():,.0f}). clean_bookings() removes the originals: {handled}.")

[bookings] 568,007 rows | cancelled dropped: 0 | superseded-by-reschedule dropped: 1,500
[bookings] status_at_export values: {nan: 541464, 'booked': 25043, 'rescheduled': 1500}

[PASS] 4. Reschedule double counting
    1,500 bookings carry rescheduled_from (expected 1,500); originals found in bookings 100%; replacement has the same test count as its original in 100% of pairs; originals hold 3,542 lines (0.27% of volume) and 1,242 invoices (INR 2,163,302). clean_bookings() removes the originals: True.


### Check 5. Ghost results

`orphans_results` holds results whose booking does not exist anywhere. We confirm they are in neither `bookings` nor `booking_lines`, and that the pipeline never loads them.

**What we used:** `.str.extract(regex)`, `value_counts().sort_index()`.

In [45]:
o_in_bk = int(orph["booking_id"].isin(bk["booking_id"]).sum())
o_in_ln = int(orph["booking_id"].isin(ln["booking_id"]).sum())
o_month = orph["booking_id"].str.extract(r"-(\d{6})-")[0].value_counts().sort_index()
add("5. Ghost results", "PASS" if o_in_bk == 0 and o_in_ln == 0 else "FLAG",
    f"{len(orph):,} rows / {orph['booking_id'].nunique():,} IDs (expected {EXPECT['orphan_results']:,}); found in bookings "
    f"{o_in_bk:,}, in booking_lines {o_in_ln:,}; ID months {o_month.index.min()}..{o_month.index.max()}. "
    f"The pipeline never loads orphans_*, so they stay out of volume and TAT.")


[PASS] 5. Ghost results
    5,401 rows / 5,399 IDs (expected 5,401); found in bookings 0, in booking_lines 0; ID months 202501..202606. The pipeline never loads orphans_*, so they stay out of volume and TAT.


### Check 6. Bookings without a collection record

A missing collection record is **unknown**, not a no-show. If many "unknown" bookings are invoiced, the record is missing, not the visit. Check 6b: a no-show must never be invoiced.

**What we used:** `groupby().mean()` on a True/False column (share invoiced), `~isin`.

In [46]:
unk = bk["coll_status"].eq("unknown")
inv_share = bk.groupby("coll_status")["invoiced"].mean().round(3).to_dict()
col_orphan = int((~col["booking_id"].isin(bk["booking_id"])).sum())
add("6. Bookings without a collection record", "FLAG" if inv_share.get("unknown", 0) > 0.5 else "INFO",
    f"bookings {len(bk):,} (expected {EXPECT['bookings']:,}) vs collections {len(col):,} (expected {EXPECT['collections']:,}); "
    f"unknown = {int(unk.sum()):,} (expected gap {EXPECT['gap']:,}); collection rows with no booking {col_orphan:,}. "
    f"Unknown by system {bk[unk].groupby('is_nova').size().to_dict()}, superseded among unknown {int(bk.loc[unk, 'superseded'].sum()):,}. "
    f"Invoiced share by status {inv_share}. Label these 'unknown'; a high invoiced share means a missing record, not an absence.")

nsb = bk["coll_status"].eq("no_show")
add("6b. No-show => no collection => no invoice", "PASS" if bk.loc[nsb, "invoiced"].sum() == 0 else "FLAG",
    f"no_show bookings {int(nsb.sum()):,}; of which invoiced {int(bk.loc[nsb, 'invoiced'].sum()):,}.")


[INFO] 6. Bookings without a collection record
    bookings 568,007 (expected 568,007) vs collections 549,598 (expected 549,598); unknown = 18,409 (expected gap 18,409); collection rows with no booking 0. Unknown by system {False: 16191, True: 2218}, superseded among unknown 51. Invoiced share by status {'collected': 1.0, 'no_show': 0.0, 'unknown': 0.019}. Label these 'unknown'; a high invoiced share means a missing record, not an absence.

[PASS] 6b. No-show => no collection => no invoice
    no_show bookings 79,172; of which invoiced 0.


### Check 7. Grain: lines vs bookings

Test volume = number of rows in `booking_lines`. Bookings, patients and invoices are **different grains** and must not be mixed up.

**What we used:** `groupby().size()`, `.is_unique`, `.mean()`, `.max()`.

In [47]:
lpb = ln.groupby("booking_id").size()
add("7. Grain: lines vs bookings", "PASS" if len(ln) == EXPECT["lines"] else "FLAG",
    f"booking_lines {len(ln):,} (expected {EXPECT['lines']:,}) across {lpb.size:,} bookings (mean {lpb.mean():.2f}, max {lpb.max()}); "
    f"bookings with zero lines {int((bk['tests'] == 0).sum()):,}; lines whose booking is missing "
    f"{int((~ln['booking_id'].isin(bk['booking_id'])).sum()):,}; line id unique: "
    f"{ln[LID].is_unique if LID else 'n/a'}. DEFINITION: test volume = booking_lines rows; patients/bookings/invoices are other grains.")


[PASS] 7. Grain: lines vs bookings
    booking_lines 1,315,204 (expected 1,315,204) across 568,007 bookings (mean 2.32, max 6); bookings with zero lines 0; lines whose booking is missing 0; line id unique: n/a. DEFINITION: test volume = booking_lines rows; patients/bookings/invoices are other grains.


### Check 8. Discounts are not in the line prices

8a: line price should equal list price (no discount stored in lines).
8b: the invoice should equal tests_total + home_fee - waiver. We measure how many do not, whether they are below or above expected, and whether the mismatch rate changes by month.

**What we used:** two chained `merge`s, `.assign(...)` inside a `groupby`, splitting a table into `neg` and `pos`.

In [48]:
pv_vals = ln[PV].value_counts(dropna=False).head(3).to_dict() if PV else "n/a"
same_price = (ln["test_price"] == ln["list_price_rs"]).mean()
add("8a. Line prices carry no discount", "PASS" if (PV and ln[PV].nunique() == 1) else "FLAG",
    f"price_vs_list_rs values {pv_vals}; test_price == list_price_rs on {same_price:.1%} of lines.")

x = inv.merge(bk[["booking_id", "tests_total", "home_fee"]], on="booking_id", how="left").merge(camp, on="booking_id", how="left")
x["expected"] = x["tests_total"] + x["home_fee"] - x["waiver_rs"].fillna(0)
x["diff"] = x["amount"] - x["expected"]
mis = x[x["diff"].abs() > 1]
neg, pos = mis[mis["diff"] < 0], mis[mis["diff"] > 0]
by_m = (x.assign(m=x["invoice_date"].dt.to_period("M").astype(str), bad=x["diff"].abs() > 1)
          .groupby("m")["bad"].mean().round(3))
add("8b. Invoice-level discount / reconciliation", "FLAG" if len(mis) / len(x) > 0.02 else "PASS",
    f"{1 - len(mis) / len(x):.1%} of invoices equal tests_total + home_fee - waiver; {len(mis):,} do not "
    f"(below expected {len(neg):,}, mean INR {neg['diff'].mean():,.0f}; above expected {len(pos):,}, mean INR {pos['diff'].mean():,.0f}). "
    f"Mismatch rate by invoice month min/max {by_m.min():.1%}/{by_m.max():.1%}. Discounts live here, not in lines.")


[PASS] 8a. Line prices carry no discount
    price_vs_list_rs values {0.0: 1313893, nan: 1311}; test_price == list_price_rs on 99.9% of lines.

[FLAG] 8b. Invoice-level discount / reconciliation
    90.9% of invoices equal tests_total + home_fee - waiver; 42,713 do not (below expected 0, mean INR nan; above expected 42,713, mean INR 150). Mismatch rate by invoice month min/max 0.0%/51.4%. Discounts live here, not in lines.


### Check 9a. Date coverage

Invoice dates should never be earlier than the date the booking was created.

**What we used:** `.dt.normalize()` (drops the time), comparing two date columns.

In [49]:
ib = inv.merge(bk[["booking_id", "created_at"]], on="booking_id")
early = int((ib["invoice_date"] < ib["created_at"].dt.normalize()).sum())
add("9a. Date coverage", "FLAG" if early else "PASS",
    f"created_at {bk['created_at'].min():%Y-%m-%d}..{bk['created_at'].max():%Y-%m-%d}; slot_at {bk['slot_at'].min():%Y-%m-%d}.."
    f"{bk['slot_at'].max():%Y-%m-%d}; invoice_date {inv['invoice_date'].min():%Y-%m-%d}..{inv['invoice_date'].max():%Y-%m-%d}; "
    f"invoices dated before their booking was created: {early:,}.")


[FLAG] 9a. Date coverage
    created_at 2024-12-26..2026-07-02; slot_at 2025-01-01..2026-07-04; invoice_date 2024-12-30..2026-06-30; invoices dated before their booking was created: 1,282.


### Check 9b. Censoring of results

The newest bookings have no result yet simply because there has not been time. If we include them in "result is blank" or turnaround metrics, we get a false problem. We compare blank-result rates before and after a cut-off of (latest result date - 14 days).

**What we used:** `pd.Timedelta`, `.dt.to_period("W")` (weekly groups), boolean filters on dates.

In [50]:
if RES:
    ln["slot_at"] = ln["booking_id"].map(bk_idx["slot_at"])
    ln["coll"] = ln["booking_id"].map(bk_idx["coll_status"])
    d = ln[ln["coll"].eq("collected")].copy()          # only collected bookings can have results
    d["res_null"] = d[RES].isna()
    cutoff = ln[RES].max() - pd.Timedelta(days=CENSOR_DAYS)
    before, after = d.loc[d["slot_at"] <= cutoff, "res_null"].mean(), d.loc[d["slot_at"] > cutoff, "res_null"].mean()
    weekly = d[d["slot_at"] > ln[RES].max() - pd.Timedelta(days=42)].groupby(d["slot_at"].dt.to_period("W"))["res_null"].mean().round(3)
    weekly.index = weekly.index.start_time.strftime("wk %m-%d")
    add("9b. Censoring of result_ts", "FLAG" if after > before * 2 + 0.01 else "PASS",
        f"latest result_ts {ln[RES].max():%Y-%m-%d}; null result_ts on collected lines: {before:.1%} up to cutoff "
        f"(latest - {CENSOR_DAYS}d) vs {after:.1%} after it; last six weeks {weekly.to_dict()}. "
        f"Exclude bookings after the cutoff from TAT / result-null metrics.")
else:
    add("9b. Censoring of result_ts", "INFO", "no result_ts column in booking_lines - set RES manually.")


[PASS] 9b. Censoring of result_ts
    latest result_ts 2026-07-02; null result_ts on collected lines: 1.9% up to cutoff (latest - 14d) vs 1.8% after it; last six weeks {'wk 05-18': 0.019, 'wk 05-25': 0.018, 'wk 06-01': 0.02, 'wk 06-08': 0.019, 'wk 06-15': 0.019, 'wk 06-22': 0.018, 'wk 06-29': 0.02}. Exclude bookings after the cutoff from TAT / result-null metrics.


### Check 10. Missing test codes

About 0.1% of lines have no `test_code`. Rule: **keep them in volume, drop them from mix and price analysis**. We create `use_for_mix` for that, and show growth with and without them to prove the difference is negligible.

**What we used:** `.isna()`, `.astype(bool)`, a small function `g(df)`, `.assign`.

In [51]:
miss = ln["test_code"].isna()
flag_ok = (ln[FLAGC].astype(bool) == miss).all() if FLAGC else "n/a"
ln["use_for_mix"] = ~miss                                           # use this mask for mix / price analysis
lv = ln.assign(month=ln["booking_id"].map(bk_idx["month"]), sup=ln["booking_id"].map(bk_idx["superseded"]))
lv = lv[~lv["sup"].fillna(False)]
def g(df):
    a = df.loc[df["month"].isin([f"2025-{m:02d}" for m in range(1, 7)])].shape[0]
    b = df.loc[df["month"].isin([f"2026-{m:02d}" for m in range(1, 7)])].shape[0]
    return b / a - 1
add("10. Missing test_code", "PASS" if abs(miss.mean() - EXPECT["missing_code_pct"]) < 0.0005 and flag_ok is not False else "FLAG",
    f"{int(miss.sum()):,} lines ({miss.mean():.2%}, expected ~{EXPECT['missing_code_pct']:.2%}); test_code_missing flag matches nulls: {flag_ok}. "
    f"H1 booked-test growth incl. them {g(lv):+.2%} vs excl. {g(lv[lv['test_code'].notna()]):+.2%} - keep in volume, "
    f"exclude (ln['use_for_mix']) from mix and price.")


[PASS] 10. Missing test_code
    1,311 lines (0.10%, expected ~0.10%); test_code_missing flag matches nulls: True. H1 booked-test growth incl. them +4.97% vs excl. +4.97% - keep in volume, exclude (ln['use_for_mix']) from mix and price.


### Funnel: where do booked tests go?

Every booked test ends up in exactly one bucket:
1. superseded (rescheduled original)
2. no-show
3. unknown, not invoiced
4. unknown, invoiced
5. collected, not invoiced
6. collected and invoiced

Comparing H1-25 and H1-26 shows whether more tests are leaking before becoming revenue.

**What we used:** `np.select` with several conditions, `groupby(array)`, a custom print format with `lambda`.

In [52]:
def funnel(months):
    d = bk[bk["month"].isin(months)]
    stage = np.select(
        [d["superseded"], d["coll_status"].eq("no_show"),
         d["coll_status"].eq("unknown") & ~d["invoiced"], d["coll_status"].eq("unknown") & d["invoiced"],
         d["coll_status"].eq("collected") & ~d["invoiced"]],
        ["1 superseded (rescheduled original)", "2 no_show", "3 unknown, not invoiced", "4 unknown, invoiced",
         "5 collected, not invoiced"], default="6 collected & invoiced")
    return d.groupby(stage)["tests"].sum()

fun = pd.DataFrame({"H1-25 tests": funnel([f"2025-{m:02d}" for m in range(1, 7)]),
                    "H1-26 tests": funnel([f"2026-{m:02d}" for m in range(1, 7)])})
fun["H1-25 share"] = fun["H1-25 tests"] / fun["H1-25 tests"].sum()
fun["H1-26 share"] = fun["H1-26 tests"] / fun["H1-26 tests"].sum()
print("\n=== FUNNEL: booked tests (booking_lines grain) by what happened next ===")
print(fun.to_string(float_format=lambda v: f"{v:,.3f}" if v < 1.5 else f"{v:,.0f}"))


=== FUNNEL: booked tests (booking_lines grain) by what happened next ===
                                     H1-25 tests  H1-26 tests  H1-25 share  H1-26 share
1 superseded (rescheduled original)          NaN         3542          NaN        0.008
2 no_show                                 61,827        61730        0.142        0.134
3 unknown, not invoiced                   12,730        16837        0.029        0.036
4 unknown, invoiced                          270          270        0.001        0.001
6 collected & invoiced                   361,475       379165        0.828        0.822


### Summary table of all checks

**What we used:** `pd.DataFrame(list_of_dicts)`, selecting two columns, `to_string(index=False)`.

In [53]:
trap_report = pd.DataFrame(report)
print("\n=== SUMMARY ===")
print(trap_report[["Status", "Trap"]].to_string(index=False))


=== SUMMARY ===
Status                                        Trap
  PASS   1. 'collections' is phlebotomy, not money
  PASS                      2. Two booking systems
  PASS        3. Nova source/channel contradiction
  PASS               4. Reschedule double counting
  PASS                            5. Ghost results
  INFO     6. Bookings without a collection record
  PASS  6b. No-show => no collection => no invoice
  PASS                 7. Grain: lines vs bookings
  PASS           8a. Line prices carry no discount
  FLAG 8b. Invoice-level discount / reconciliation
  FLAG                           9a. Date coverage
  PASS                  9b. Censoring of result_ts
  PASS                       10. Missing test_code


## Part 9. Payments detective work

Now we dig into **money**: do payments match invoices, and what are the leaks? This part loads the raw tables again under short names (`bk`, `lines`, `invoices`, `payments`, ...). It deliberately **overwrites** the `bk` etc. from Part 8, so run Part 9 after Part 8 and do not mix the two.

The style here is exploratory: small cells, print, look, ask the next question.

### 9.1 Load the raw tables

**What we used:** a small `load()` function, `os.listdir` to confirm files exist.

In [54]:
import os
import pandas as pd

D = DATA_DIR

# 1. Confirm the folder and file names before loading
print(os.path.isdir(D))
print(os.listdir(D))

# 2. Load
def load(name):
    return pd.read_csv(os.path.join(D, name))

bk          = load("bookings.csv.gz")
lines       = load("booking_lines.csv.gz")
invoices    = load("invoices.csv.gz")
payments    = load("payments.csv.gz")
campaigns   = load("campaigns.csv.gz")
collections = load("collections.csv.gz")
plan        = load("plan_long.csv.gz")

True
['orphans_results.csv.gz', 'orphans_collections.csv.gz', 'orphans_invoices.csv.gz', 'booking_lines.csv.gz', 'bookings.csv.gz', 'collections.csv.gz', 'invoices.csv.gz', 'plan_long.csv.gz', 'campaigns.csv.gz', 'payments.csv.gz']


/tmp/ipykernel_9713/943483342.py:12: DtypeWarning: Columns (8,11,12,13) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(os.path.join(D, name))
/tmp/ipykernel_9713/943483342.py:12: DtypeWarning: Columns (8,13,14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(os.path.join(D, name))


### 9.2 Does the invoice formula hold? Which discount formula is right?

Two candidate formulas for the expected invoice:
- **A:** tests_total + home_fee - waiver (waiver is *not* inside home_fee)
- **B:** tests_total + home_fee (waiver is *already inside* home_fee)

We count the mismatches for campaign and non-campaign bookings under each formula. The formula with near-zero mismatches wins. Then we check whether any reschedule pair was **billed twice** (original and replacement both invoiced).

**What we used:** `merge(how="left")`, `.notna()`, `pd.crosstab`, a loop over formula names, a double merge on the same table with `suffixes`.

(The loop variable is renamed `fm`; the original used `f`, which would overwrite our fact table `f` from Part 6.)

In [55]:
# Step 0: what does home_fee look like on campaign bookings?
chk = bk.merge(campaigns[['booking_id','waiver_rs']], on='booking_id', how='left')
chk['in_campaign'] = chk.waiver_rs.notna()
print(pd.crosstab([chk.in_campaign, chk.channel], chk.home_fee))

# Mismatch test with both candidate formulas (raw bk, so superseded originals stay in)
inv = (invoices
       .merge(bk[['booking_id','tests_total','home_fee','channel']], on='booking_id')
       .merge(campaigns[['booking_id','waiver_rs']], on='booking_id', how='left'))
inv['in_campaign'] = inv.waiver_rs.notna()

inv['exp_A'] = inv.tests_total + inv.home_fee - inv.waiver_rs.fillna(0)   # waiver not in home_fee
inv['exp_B'] = inv.tests_total + inv.home_fee                              # waiver already in home_fee

for fm in ['exp_A', 'exp_B']:
    inv['miss_' + fm] = (inv.amount - inv[fm]).abs() > 0.5
    print(fm)
    print(pd.crosstab(inv.in_campaign, inv['miss_' + fm]), '\n')

# Reschedule double billing
orig = bk.loc[bk.rescheduled_from.notna(), ['booking_id', 'rescheduled_from']]
both = (orig.merge(invoices, on='booking_id')
            .merge(invoices, left_on='rescheduled_from', right_on='booking_id',
                   suffixes=('_new', '_old')))
print(len(both), both.amount_old.sum())

home_fee                0       150
in_campaign channel                
False       clinic   353600       0
            home          0  165538
True        home      48869       0
exp_A
miss_exp_A    False  True 
in_campaign               
False        428063     18
True              0  42695 

exp_B
miss_exp_B    False  True 
in_campaign               
False        428063     18
True          42695      0 

0 0.0


### 9.3 Reschedule structure: who is collected, who is invoiced?

For rescheduled bookings: which system are they from, how many replacements vs originals have an invoice, and which of them appear in `collections`? This tells us the visit and the invoice sit on the **original**, not the replacement.

**What we used:** `set` operations (`&` for overlap), `isin().sum()`.

In [56]:
r = bk.loc[bk.rescheduled_from.notna(), ['booking_id','rescheduled_from','source_system','status_at_export']]
print(r.head(), '\n', r.source_system.value_counts(), r.status_at_export.value_counts(dropna=False))

orig_ids = set(r.rescheduled_from)
repl_ids = set(r.booking_id)
inv_ids  = set(invoices.booking_id)
print('replacements invoiced:', len(repl_ids & inv_ids), 'of', len(repl_ids))
print('originals invoiced   :', len(orig_ids & inv_ids), 'of', len(orig_ids))
print('originals in collections:', collections.booking_id.isin(orig_ids).sum(),
      '| replacements in collections:', collections.booking_id.isin(repl_ids).sum())

              booking_id  rescheduled_from source_system status_at_export
547033  nova_che_0093125  nova_che_0090017          nova           booked
547034  nova_che_0093126  nova_che_0092841          nova           booked
547035  nova_che_0093127  nova_che_0088334          nova           booked
547036  nova_che_0093128  nova_che_0091525          nova           booked
547037  nova_che_0093129  nova_che_0093005          nova           booked 
 source_system
nova    1500
Name: count, dtype: int64 status_at_export
booked    1500
Name: count, dtype: int64
replacements invoiced: 0 of 1500
originals invoiced   : 1242 of 1500
originals in collections: 1449 | replacements in collections: 0


### 9.4 Match payments to invoices (fix the broken IDs)

Payment IDs come in messy forms (`INV2026090659`, `INV-2026-090659`, ...). We:
1. strip every non-alphanumeric character
2. rebuild the standard form `INV-YYYY-NNNNNN` with a regex that captures two groups
3. compare the share that matches invoices before and after normalising
4. compare cash captured per invoice with the invoiced amount: under / exact / over

**What we used:** `.str.replace` with regex and group references (`\1`, `\2`), `.isin`, `groupby().sum()`, `.join`, `pd.cut` to bucket differences.

In [57]:
import re
p = payments.copy()
p['inv_clean'] = p.invoice_id_raw.str.replace(r'[^A-Za-z0-9]', '', regex=True)
p['inv_norm']  = p.inv_clean.str.replace(r'^INV(\d{4})(\d{6})$', r'INV-\1-\2', regex=True)
print('raw join match   :', p.invoice_id_raw.isin(invoices.invoice_id).mean())
print('normalised match :', p.inv_norm.isin(invoices.invoice_id).mean())

cap = p[p.payment_status == 'captured']
per_inv = cap.groupby('inv_norm').amount_rs.sum().rename('paid')
x = invoices.set_index('invoice_id').join(per_inv)
x['paid'] = x.paid.fillna(0)
print('invoiced Rs', x.amount.sum(), '| captured Rs', x.paid.sum())
print(pd.cut(x.paid - x.amount, [-1e9, -0.5, 0.5, 1e9], labels=['under','exact','over']).value_counts())

raw join match   : 0.6950731456979516
normalised match : 0.8917587279596711
invoiced Rs 737020069.0 | captured Rs 635333547.0
exact    402349
under     66555
over       1872
Name: count, dtype: int64


### 9.5 Settlements: does gross = net + fees + tax?

Payments are grouped into bank settlements. For each settlement we compare the gross amount captured with net + fee + tax. The summary of the absolute difference should be about zero.

**What we used:** `dropna(subset=...)`, `groupby().agg(name=(col, "first"))`, `.describe()`.

In [58]:
s = cap.dropna(subset=['settlement_id']).groupby('settlement_id').agg(
        gross=('amount_rs','sum'), net=('settlement_amount_rs','first'),
        fee=('fees_rs','first'), tax=('tax_rs','first'))
s['diff'] = s.gross - (s.net + s.fee + s.tax)
print(s['diff'].abs().describe())

count   562.000
mean      0.000
std       0.000
min       0.000
25%       0.000
50%       0.000
75%       0.000
max       0.000
Name: diff, dtype: float64


### 9.6 Reschedule volume and months

How many test lines belong to replacement bookings (the doc says about 3,542), and in which slot months do they fall?

**What we used:** `.isin`, `.shape[0]`, `value_counts().sort_index()`.

In [59]:
rep = bk.loc[bk.rescheduled_from.notna(), 'booking_id']
print(lines[lines.booking_id.isin(rep)].shape[0])                       # expect ~3,542
print(pd.to_datetime(bk[bk.booking_id.isin(rep)].slot_at).dt.to_period('M').value_counts().sort_index())

3542
slot_at
2026-04    452
2026-05    504
2026-06    500
2026-07     44
Freq: M, Name: count, dtype: int64


### 9.7 Do reschedules change the slot date?

We join each replacement to its original and compare slot dates (same date? how many days apart?).

**What we used:** double merge with `suffixes`, `.dt.days`, `.describe()`.

In [60]:
r = (bk[bk.rescheduled_from.notna()][['booking_id','rescheduled_from','slot_at','created_at']]
     .merge(bk[['booking_id','slot_at','created_at']], left_on='rescheduled_from', right_on='booking_id',
            suffixes=('_new','_old')))
print((r.slot_at_new == r.slot_at_old).mean())
print((pd.to_datetime(r.slot_at_new) - pd.to_datetime(r.slot_at_old)).dt.days.describe())

0.0
count   1,500.000
mean        2.479
std         1.118
min         1.000
25%         1.000
50%         2.000
75%         3.000
max         4.000
dtype: float64


### 9.8 How big are gateway fees and tax?

Total fees, tax and net, and fees plus tax as a share of gross.

In [61]:
print(s.fee.sum(), s.tax.sum(), s.net.sum(), (s.fee.sum()+s.tax.sum())/s.gross.sum())

14249782.66 2564960.9499999997 695674389.39 0.023600000099931347


### 9.9 Final payment-to-invoice mapping and the "buckets"

Some payment IDs still do not match even after normalising. For those we fall back to `booking_id`. Then each invoice is put in one bucket:

| Bucket | Meaning |
|---|---|
| exact | paid = invoiced |
| over | paid more than invoiced (duplicate capture?) |
| partial | paid less than invoiced |
| failed_only | only failed attempts exist |
| no_payment_row | no payment record at all |

We also count captured payments that map to no invoice, and invoices captured more than once.

**What we used:** `Series.where(cond, other)` (use this value if cond, else another), `np.select` for buckets, `groupby().agg()`, `.fillna(0)`.

In [62]:
import numpy as np
p = payments.copy()
p['inv_norm'] = (p.invoice_id_raw.str.replace(r'[^A-Za-z0-9]', '', regex=True)
                  .str.replace(r'^INV(\d{4})(\d{6})$', r'INV-\1-\2', regex=True))
inv_ids = set(invoices.invoice_id)
p['m_norm']  = p.inv_norm.isin(inv_ids)
p['m_clean'] = p.invoice_id.isin(inv_ids)
p['m_bk']    = p.booking_id.isin(set(invoices.booking_id))
print(p[['m_norm','m_clean','m_bk']].value_counts(dropna=False))
print(p.loc[~p.m_norm, ['invoice_id','invoice_id_raw','booking_id','payment_status','amount_rs']].sample(10, random_state=1))

# final mapping: id first, else booking_id
bk2inv = invoices.set_index('booking_id').invoice_id
p['inv_final'] = p.inv_norm.where(p.m_norm, p.booking_id.map(bk2inv))
cap = p[p.payment_status == 'captured']
failed_set = set(p.loc[p.payment_status == 'failed', 'inv_final'].dropna())
print('captured unmapped:', cap.inv_final.isna().sum(), cap.loc[cap.inv_final.isna(), 'amount_rs'].sum())

paid = cap.groupby('inv_final').amount_rs.agg(paid='sum', n_cap='count')
x = invoices.set_index('invoice_id').join(paid)
x[['paid','n_cap']] = x[['paid','n_cap']].fillna(0)
x['gap'] = x.amount - x.paid
x['bucket'] = np.select(
    [x.paid.sub(x.amount).abs() <= 0.5, x.paid > x.amount + 0.5,
     (x.paid > 0) & (x.paid < x.amount - 0.5), x.index.isin(failed_set) & (x.paid == 0), x.paid == 0],
    ['exact', 'over', 'partial', 'failed_only', 'no_payment_row'], 'other')
print(x.groupby('bucket').agg(n=('amount','size'), invoiced=('amount','sum'), paid=('paid','sum'), gap=('gap','sum')))

print('captured total', cap.amount_rs.sum(),
      '| captured w/o settlement', cap.settlement_id.isna().sum(), cap.loc[cap.settlement_id.isna(), 'amount_rs'].sum())
print('multi-capture invoices:', (x.n_cap > 1).sum())

m_norm  m_clean  m_bk 
True    True     True     422254
False   True     True      46488
        False    False      4765
Name: count, dtype: int64
              invoice_id    invoice_id_raw            booking_id payment_status  amount_rs
167011   INV-2025-164717   inv-2025-164717  BK-HYD-202507-033026       captured  1,700.000
10917    INV-2025-010783   inv-2025-010783  BK-AHM-202501-000155         failed  1,450.000
162802   INV-2025-160577   inv-2025-160577  BK-PUN-202507-026371       captured  2,569.000
219425  INV-2025-1116354  INV-2025-1116354                   NaN         failed    670.000
445464   INV-2026-132904   inv-2026-132904      nova_jai_0049891       captured    350.000
414627   INV-2026-101519   inv-2026-101519  BK-AHM-202604-085781       captured    280.000
263959  INV-2025-1160328  INV-2025-1160328                   NaN       captured    650.000
387160   INV-2026-074353   inv-2026-074353  BK-HYD-202603-075848       captured    650.000
84551    INV-2025-083423   inv-20

### 9.10 When are reschedule originals collected, relative to the slots?

Hours between the collection time and the original slot / the new slot. It shows whether the visit really happened at the original time.

**What we used:** `.dt.total_seconds().div(3600)` (seconds to hours), filtering on a status.

In [63]:
r = (bk[bk.rescheduled_from.notna()][['booking_id','rescheduled_from','slot_at']]
       .merge(bk[['booking_id','slot_at']], left_on='rescheduled_from', right_on='booking_id', suffixes=('_new','_old'))
       .merge(collections[['booking_id','status','collected_at']], left_on='rescheduled_from', right_on='booking_id'))
for c in ['slot_at_new','slot_at_old','collected_at']: r[c] = pd.to_datetime(r[c])
col = r[r.status == 'collected']
print((col.collected_at - col.slot_at_old).dt.total_seconds().div(3600).describe())
print((col.collected_at - col.slot_at_new).dt.total_seconds().div(3600).describe())

count   1,239.000
mean        0.498
std         1.775
min       -48.000
25%         0.117
50%         0.567
75%         1.050
max         1.483
dtype: float64
count   1,239.000
mean      -58.911
std        27.207
min      -144.000
25%       -94.533
50%       -48.200
75%       -24.283
max       -22.517
dtype: float64


### 9.11 Can orphan payments be re-attached? What are the unpaid invoices?

**a) Re-attach orphans.** An orphan payment may have a typo (one extra digit in the invoice number). We try deleting each digit in turn, and accept a candidate only if the **amount also matches** an unpaid invoice. Two independent signals make the match safe.

**b) Who are the unpaid invoices?** Their split by channel, city and source system.

**c) Duplicate captures.** Time gap between two captures on the same invoice, and whether the payment method differs.

**What we used:** a dictionary of sets (`setdefault`), `DataFrame.apply(func, axis=1)` row by row, `rsplit`, string slicing, `diff()` inside a `groupby`.

In [64]:
# a) Can the orphans be re-attached? Delete one digit, require amount to match an unpaid invoice
unpaid = x[x.bucket == 'no_payment_row'].reset_index()
cands = {}
for iid, amt in zip(unpaid.invoice_id, unpaid.amount):
    cands.setdefault(amt, set()).add(iid)
orph = cap[cap.inv_final.isna()].copy()
def fix(row):
    s = row.invoice_id
    pre, seq = s.rsplit('-', 1)
    for i in range(len(seq)):
        c = f"{pre}-{seq[:i] + seq[i+1:]}"
        if c in cands.get(row.amount_rs, ()):
            return c
orph['fixed'] = orph.apply(fix, axis=1)
print(orph.fixed.notna().sum(), orph.loc[orph.fixed.notna(), 'amount_rs'].sum())

# b) What are the 21,820 unpaid invoices?
up = unpaid.merge(bk[['booking_id','source_system','clinic_id']], on='booking_id', how='left')
for c in ['channel','city','source_system']:
    print(up.groupby(c).amount.agg(['size','sum']).assign(share=lambda d: d['size']/len(up)).round(3))
up['m'] = pd.to_datetime(up.invoice_date).dt.to_period('M')
print((up.groupby('m').size() / invoices.assign(m=pd.to_datetime(invoices.invoice_date).dt.to_period('M')).groupby('m').size()).round(3))

# c) Duplicate captures: gap between the two captures, same method?
dup = cap[cap.inv_final.isin(x.index[x.n_cap > 1])].sort_values(['inv_final','created_at'])
dup['dt'] = pd.to_datetime(dup.created_at).groupby(dup.inv_final).diff().dt.total_seconds()
print(dup.dt.describe(), dup.groupby('inv_final').method.nunique().value_counts())

1585 2553925.0
          size            sum  share
channel                             
clinic   13090 20,251,980.000  0.600
home      8730 14,569,621.000  0.400
           size           sum  share
city                                
Ahmedabad  3939 6,453,241.000  0.181
Bengaluru  5072 8,184,947.000  0.232
Chennai    3874 6,115,375.000  0.178
Hyderabad  3637 5,679,379.000  0.167
Jaipur     2104 3,341,137.000  0.096
Pune       3194 5,047,522.000  0.146
                size            sum  share
source_system                             
legacy         20126 31,958,802.000  0.922
nova            1694  2,862,799.000  0.078
m
2024-12     NaN
2025-01   0.039
2025-02   0.039
2025-03   0.038
2025-04   0.039
2025-05   0.038
2025-06   0.040
2025-07   0.038
2025-08   0.037
2025-09   0.040
2025-10   0.039
2025-11   0.039
2025-12   0.041
2026-01   0.038
2026-02   0.038
2026-03   0.040
2026-04   0.040
2026-05   0.039
2026-06   0.170
Freq: M, dtype: float64
count       1,925.000
mean      397,859

### 9.12 Unpaid *rates* by group, the June cut-off, duplicates vs reschedules

We compare **rates** (not counts), because big groups have bigger counts.

- unpaid rate by source system, channel, city and "reschedule original"
- do payments stop earlier than invoices in June? What is the normal payment lag?
- are duplicate captures linked to reschedule originals?
- what do the remaining orphan IDs look like?
- were reschedules created before or after the visit?

**What we used:** `groupby().agg(['size','sum','mean'])`, `.dt.to_period("W")`, percentiles in `describe`, set overlaps.

In [65]:
# a) Unpaid RATES by stratum, and whether reschedule originals explain the nova excess
x2 = (x.reset_index()
        .merge(bk[['booking_id','source_system','clinic_id']], on='booking_id', how='left'))
x2['unpaid'] = x2.bucket.eq('no_payment_row')
x2['orig_resched'] = x2.booking_id.isin(set(bk.rescheduled_from.dropna()))
for c in ['source_system','channel','city','orig_resched']:
    print(x2.groupby(c).unpaid.agg(['size','sum','mean']).round(4))
print(x2[~x2.orig_resched].groupby('source_system').unpaid.mean())   # nova without originals

# b) June cut-off: do payments end before invoices? what is the normal pay lag?
print(pd.to_datetime(payments.created_at).max(), pd.to_datetime(invoices.invoice_date).max())
j = x2[pd.to_datetime(x2.invoice_date).dt.to_period('M') == '2026-06'].copy()
j['wk'] = pd.to_datetime(j.invoice_date).dt.to_period('W')
print(j.groupby('wk').unpaid.agg(['size','mean']).round(3))
lag = (cap.assign(c=pd.to_datetime(cap.created_at))
          .merge(invoices[['invoice_id','invoice_date']], left_on='inv_final', right_on='invoice_id'))
print((lag.c - pd.to_datetime(lag.invoice_date)).dt.days.describe(percentiles=[.5,.9,.99]))

# c) Duplicate captures vs reschedule originals
orig_inv = set(invoices.loc[invoices.booking_id.isin(set(bk.rescheduled_from.dropna())), 'invoice_id'])
dup_inv = set(x.index[x.n_cap > 1])
dm = set(dup.groupby('inv_final').method.nunique().loc[lambda s: s == 2].index)
print(len(dup_inv & orig_inv), len(dup_inv), len(dm & orig_inv), len(dm))
print(payments.method.value_counts())   # is cash among the 4 methods?

# d) Remaining orphans: what do they look like?
rest = orph[orph.fixed.isna()]
print(rest.invoice_id.str.rsplit('-', n=1).str[1].str.len().value_counts())
print(rest.invoice_id.str[:8].value_counts().head())
print(rest.amount_rs.describe())

# e) Reschedule: created before or after the visit? And confirm the 210 no-shows
r2 = (bk[bk.rescheduled_from.notna()][['booking_id','rescheduled_from','created_at']]
        .merge(collections[['booking_id','status','collected_at']],
               left_on='rescheduled_from', right_on='booking_id', suffixes=('','_o')))
print(r2.status.value_counts())
r2['created_at'] = pd.to_datetime(r2.created_at); r2['collected_at'] = pd.to_datetime(r2.collected_at)
c = r2[r2.status == 'collected']
print((c.created_at - c.collected_at).dt.total_seconds().div(3600).describe())

                 size    sum  mean
source_system                     
legacy         449975  20126 0.045
nova            20801   1694 0.081
           size    sum  mean
channel                     
clinic   284248  13090 0.046
home     186528   8730 0.047
             size   sum  mean
city                         
Ahmedabad   84418  3939 0.047
Bengaluru  108317  5072 0.047
Chennai     86159  3874 0.045
Hyderabad   79316  3637 0.046
Jaipur      45951  2104 0.046
Pune        66615  3194 0.048
                size    sum  mean
orig_resched                     
False         469534  21718 0.046
True            1242    102 0.082
source_system
legacy   0.045
nova     0.081
Name: unpaid, dtype: float64
2026-07-15 02:19:13 2026-06-30 00:00:00
                       size  mean
wk                               
2026-06-01/2026-06-07  6210 0.039
2026-06-08/2026-06-14  6073 0.038
2026-06-15/2026-06-21  6138 0.035
2026-06-22/2026-06-28  6257 0.445
2026-06-29/2026-07-05  1679 0.606
count   450,881.0

### 9.13 Is the late-June jump real, or just missing data?

Three tests:
1. **Composition:** is the nova excess just a mix effect?
2. **The tail (from 22 June):** is the high unpaid rate a real incident or truncated payment data? We check payment counts per day against invoices per day.
3. **Steady-state leak:** in the mature period, what predicts unpaid? Amount bucket, weekday, patient repeat behaviour. We compare patients with 3+ unpaid invoices to what a pure 4% random leak would give (binomial).

**What we used:** `pd.qcut` (equal-size buckets), `groupby(dayofweek)`, `scipy.stats.binom.cdf`.

In [66]:
from scipy import stats

x2['d'] = pd.to_datetime(x2.invoice_date)
x2['is_tail'] = x2['d'] >= '2026-06-22'

# 1) Nova: composition effect?
print(x2.groupby(['source_system', 'is_tail']).unpaid.agg(['size', 'mean']).round(3))
print(x2[x2.source_system == 'nova']['d'].dt.to_period('M').value_counts().sort_index())

# 2) Tail: truncation, or a real incident?
t = x2[x2['is_tail']]
for c in ['channel', 'city', 'source_system']:
    print(t.groupby(c).unpaid.agg(['size', 'mean']).round(3))
print(t.groupby('clinic_id').unpaid.mean().describe())

pay_d = pd.to_datetime(payments.created_at).dt.normalize().value_counts().sort_index()
inv_d = pd.to_datetime(invoices.invoice_date).value_counts().sort_index()
print(pd.concat([inv_d.rename('inv'), pay_d.rename('pay')], axis=1)
        .loc['2026-06-10':].fillna(0).astype(int))

tl = lag[pd.to_datetime(lag.invoice_date) >= '2026-06-22']
print((tl.c - pd.to_datetime(tl.invoice_date)).dt.days.describe())

# 3) Steady-state leak: what predicts unpaid? (mature period only)
mt = x2[~x2['is_tail']].copy()
mt['amt_q'] = pd.qcut(mt.amount, 5, duplicates='drop')
print(mt.groupby('amt_q', observed=True).unpaid.mean().round(4))
print(mt.groupby(mt['d'].dt.dayofweek).unpaid.mean().round(4))

pp = mt.groupby('patient_id').unpaid.agg(['size', 'sum'])
exp3 = (1 - stats.binom.cdf(2, pp['size'], 0.04)).sum()
print((pp['sum'] >= 3).sum(), 'patients with 3+ unpaid; expected at p=0.04 ~', int(exp3))

                         size  mean
source_system is_tail              
legacy        False    444121 0.039
              True       5854 0.482
nova          False     18719 0.038
              True       2082 0.470
d
2026-03       3
2026-04    6933
2026-05    6895
2026-06    6970
Freq: M, Name: count, dtype: int64
         size  mean
channel            
clinic   4584 0.474
home     3352 0.485
           size  mean
city                 
Ahmedabad  1493 0.470
Bengaluru  1823 0.493
Chennai    1336 0.472
Hyderabad  1387 0.478
Jaipur      746 0.468
Pune       1151 0.485
               size  mean
source_system            
legacy         5854 0.482
nova           2082 0.470
count   31.000
mean     0.480
std      0.035
min      0.405
25%      0.457
50%      0.480
75%      0.502
max      0.559
Name: unpaid, dtype: float64
            inv  pay
2026-06-10  908  820
2026-06-11  910  944
2026-06-12  850  935
2026-06-13  866  886
2026-06-14  852  895
2026-06-15  865  883
2026-06-16  910  903
2026-0

### 9.14 Pin down the step date, check clinic noise, check the nova go-live

- **A)** the exact day the unpaid rate steps up, any surge in failed payments, and the payment-method mix by week (does cash disappear?)
- **B)** is the steady 3.9% leak pure noise at clinic level? If the spread of clinic rates equals the spread expected by chance, yes
- **C)** does total volume dip around the nova cutover, by city and week?

**What we used:** `normalize()`, `pd.crosstab(normalize="index")`, standard deviation vs `np.sqrt(p(1-p)/n)`, `groupby().min()` for first date.

In [67]:
# A) Pin the step date; any failed-payment surge? Is cash what disappears?
x2['dn'] = x2['d'].dt.normalize()
print(x2.groupby('dn').unpaid.mean().loc['2026-06-18':].round(3))
p['dn'] = pd.to_datetime(p.created_at).dt.normalize()
print(p[p.payment_status == 'failed'].groupby('dn').size().loc['2026-06-10':])
p['wk'] = pd.to_datetime(p.created_at).dt.to_period('W')
print(pd.crosstab(p.wk, p.method, normalize='index').round(3).loc['2026-05-18':])

# B) Is the steady 3.9% pure noise at clinic level?
c = mt.groupby('clinic_id').unpaid.agg(['size', 'mean'])
print(c['mean'].std(), np.sqrt(0.039 * 0.961 / c['size']).mean())   # equal -> pure noise

# C) Nova go-live: does total volume dip around cutover, by city? (relevant to Briefs A/B)
b = bk.copy(); b['slot_at'] = pd.to_datetime(b.slot_at)
b = b[(b.slot_at >= '2026-03-01') & (b.slot_at < '2026-07-01')]
print(b.groupby([b.slot_at.dt.to_period('W'), 'source_system']).size().unstack())
print(b.groupby(['city', b.slot_at.dt.to_period('M'), 'source_system']).size().unstack())
print(bk[bk.source_system == 'nova'].groupby('city').slot_at.min())

dn
2026-06-18   0.037
2026-06-19   0.034
2026-06-20   0.027
2026-06-21   0.039
2026-06-22   0.039
2026-06-23   0.042
2026-06-24   0.616
2026-06-25   0.621
2026-06-26   0.603
2026-06-27   0.612
2026-06-28   0.594
2026-06-29   0.602
2026-06-30   0.611
Name: unpaid, dtype: float64
dn
2026-06-10    22
2026-06-11    28
2026-06-12    39
2026-06-13    30
2026-06-14    31
2026-06-15    30
2026-06-16    36
2026-06-17    40
2026-06-18    23
2026-06-19    37
2026-06-20    34
2026-06-21    31
2026-06-22    28
2026-06-23    34
2026-06-24    33
2026-06-25    17
2026-06-26    23
2026-06-27    16
2026-06-28    18
2026-06-29    20
2026-06-30    20
2026-07-01    19
2026-07-02     8
2026-07-03    10
2026-07-04     4
2026-07-05     8
2026-07-06     8
2026-07-07     6
2026-07-08     4
2026-07-09     7
2026-07-12     1
2026-07-13     3
2026-07-14     1
2026-07-15     1
dtype: int64
method                 card  cash  netbanking   upi
wk                                                 
2026-05-18/2026-05-24 0

### 9.15 Orphan captures by day

When do the unmapped captured payments appear? If they cluster after a date, it points to a systems issue from that date.

In [68]:
orph_d = cap[cap.inv_final.isna()].assign(dn=lambda d: pd.to_datetime(d.created_at).dt.normalize())
print(orph_d.groupby('dn').size().loc['2026-06-15':])

dn
2026-06-15     9
2026-06-16    13
2026-06-17    11
2026-06-18    15
2026-06-19     8
2026-06-20     6
2026-06-21     6
2026-06-22     4
2026-06-23    12
2026-06-24     4
2026-06-26     6
2026-06-27     4
2026-06-28     2
2026-06-29     6
2026-06-30     1
2026-07-01     3
2026-07-02     6
2026-07-03     6
2026-07-04     3
2026-07-05     2
2026-07-07     1
2026-07-08     1
2026-07-14     1
dtype: int64


### 9.16 Is any clinic unusual? (chi-square idea)

For each clinic we compute its unpaid rate `p`, the overall rate `p0`, and a **z-score**: how many standard errors away. The sum of squared z-scores compared with the number of clinics (minus 1) tells us whether the spread is bigger than chance. If they are about equal, no clinic is special.

**What we used:** z-score formula, `sort_values().iloc[[...]]` to see the extremes.

In [69]:
c = mt.groupby('clinic_id').unpaid.agg(n='size', k='sum'); c['p'] = c.k / c.n
p0 = c.k.sum() / c.n.sum()
c['z'] = (c.p - p0) / np.sqrt(p0 * (1 - p0) / c.n)
print((c.z ** 2).sum(), len(c) - 1)
print(c.sort_values('z').iloc[[0, 1, 2, -3, -2, -1]])

41.9494887268191 30
               n     k     p      z
clinic_id                          
HYD-02     17815   630 0.035 -2.463
HYD-04     10724   375 0.035 -2.123
CHE-01     28857  1062 0.037 -1.872
BLR-03     17834   736 0.041  1.613
PUN-01     22411   924 0.041  1.777
PUN-03     11874   513 0.043  2.405


## Recap: what we used, and what each part answers

| Part | Question answered | Main tools |
|---|---|---|
| 2 to 3 | Is the base data clean and joinable? | `drop_duplicates`, `to_datetime`, `isin`, `merge`, `np.select` |
| 4 | What drives revenue growth? | `groupby().agg`, `nunique`, `np.log`, `assert` |
| 5 | Why did revenue per test move? Is cash collected? | `map`, `drop_duplicates`, left `merge` |
| 7 | What does the plan's "18%" refer to? | dict of Series, `isin` on index, `idxmin` |
| 8 | Which dictionary claims hold? | helper functions, `crosstab`, `np.select`, PASS/FLAG/INFO |
| 9 | Do invoices, payments and settlements agree? | normalising IDs with regex, `pd.cut`, `qcut`, z-scores |

**Honest caveats**
- The code is the original logic, re-cut into small cells. It has not been run here because the data was not available.
- Part 8 and Part 9 both create variables called `bk`, `inv`, `p`, `x`. They are different objects in each part. If you re-run only part of the notebook, re-run the cell that loads the data for that part first.